In [ ]:
!pip install ollama

In [ ]:
!apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess, time, httpx

subprocess.Popen(["ollama", "serve"])

for i in range(10):
    time.sleep(2)
    try:
        httpx.get("http://localhost:11434", timeout=3)
        print("server is up ✓")
        break
    except Exception:
        print(f"waiting... ({i+1})")

server is up ✓


In [ ]:
!ollama pull qwen3.5:2b
!ollama pull gemma2:2b

In [ ]:
!ollama list

NAME          ID              SIZE      MODIFIED               
gemma2:2b     8ccf136fdd52    1.6 GB    Less than a second ago    
qwen3.5:2b    324d162be6ca    2.7 GB    Less than a second ago    


In [ ]:
# ============================================================
# مستشار الوظائف التفاعلي — Ollama في Colab
# ============================================================

import ollama                              # مكتبة الاتصال بسيرفر Ollama
from IPython.display import display, HTML  # لعرض HTML داخل مخرجات الخلية


# ============================================================
# القسم 0: اسم الموديل
# ============================================================

MODEL_NAME = "gemma2:2b"


# ============================================================
# القسم 1: تعليمات المستشار (System Prompt)
# ============================================================

SYSTEM = """أنت مستشار توظيف سعودي ودود ومختصر.
خاطب المستخدم بصيغة محايدة (مثل: أكتب/ي، وش مجالك).
اسأل المستخدم 5 أسئلة، سؤال واحد فقط في كل رد:
1. وش مجالك أو تخصصك؟
2. كم سنة خبرة عندك؟
3. هل عندك شهادات احترافية؟ وإذا نعم، وش هي؟
4. وش المدينة اللي تبي تشتغل فيها؟
5. وش أهم شي لك: الراتب، التعلم والتطور، أو الاستقرار؟

بعد ما تجمع الإجابات الخمس، قدم له:
- 3 مسميات وظيفية تناسبه بالضبط
- نصيحة عملية وحدة لتقوية ملفه (وإذا ما عنده شهادات، اقترح أنسب شهادة لمجاله)
- نوع الجهات اللي يستهدفها (حكومي/ستارتب/شركات كبرى)
خلك مختصر وواقعي لسوق العمل السعودي."""


# ============================================================
# القسم 2: دالة عرض الفقاعات
# ============================================================

def show_bubble(text: str, role: str) -> None:
    if role == "bot":
        bg, align, icon = "#f0f4ff", "right", "🤖"
    else:
        bg, align, icon = "#e8f5e9", "left", "👤"
    text = text.replace("\n", "<br>")
    display(HTML(f"""
    <div dir="rtl" style="
        background:{bg};
        border-radius:14px;
        padding:12px 16px;
        margin:6px 0;
        max-width:75%;
        float:{align};
        clear:both;
        font-family:'Segoe UI', Tahoma, sans-serif;
        font-size:15px;
        line-height:1.8;
        color:#1a1a1a;
        box-shadow:0 1px 2px rgba(0,0,0,0.08);
    ">{icon} {text}</div>
    <div style="clear:both"></div>
    """))


# ============================================================
# القسم 3: تهيئة الحوار
# ============================================================

messages = [{"role": "system", "content": SYSTEM}]
show_bubble("مستشار الوظائف جاهز — أكتب/ي «خلاص» للإنهاء", "bot")

messages.append({
    "role": "user",
    "content": "أهلاً، أبي استشارة وظيفية. ابدأ بأول سؤال."
})


# ============================================================
# القسم 4: حلقة الحوار الرئيسية
# ============================================================

while True:
    # 1) استدعاء الموديل بالتاريخ الكامل
    resp = ollama.chat(model=MODEL_NAME, messages=messages)
    reply = resp["message"]["content"]

    # شبكة أمان: لو رجع رد فاضي لأي سبب، نعرض تنبيه بدل فقاعة فاضية
    if not reply.strip():
        reply = "(رد فاضي من الموديل — أعيدي تشغيل الخلية أو جربي موديل ثاني)"

    # 2) عرض الرد وحفظه في الذاكرة
    show_bubble(reply, "bot")
    messages.append({"role": "assistant", "content": reply})

    # 3) إدخال المستخدم
    user_input = input("👤 أكتب/ي ردك: ")
    if user_input.strip() in ["خلاص", "exit", "quit"]:
        show_bubble("بالتوفيق! 🌟", "bot")
        break
    show_bubble(user_input, "user")
    messages.append({"role": "user", "content": user_input})

👤 أكتب/ي ردك: ذكاء اصطناعي


👤 أكتب/ي ردك: سنة واحدة


👤 أكتب/ي ردك: لا


👤 أكتب/ي ردك: الرياض


👤 أكتب/ي ردك: الذكاء الاصطناعي


👤 أكتب/ي ردك: خلاص


In [ ]:
!ollama show gemma2:2b

  Model
    architecture        gemma2    
    parameters          2.6B      
    context length      8192      
    embedding length    2304      
    quantization        Q4_0      

  Capabilities
    completion    

  Parameters
    stop    "<start_of_turn>"    
    stop    "<end_of_turn>"      

  License
    Gemma Terms of Use                  
    Last modified: February 21, 2024    
    ...                                 



In [ ]:
!ollama ps

NAME         ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
gemma2:2b    8ccf136fdd52    1.9 GB    100% GPU     4096       3 minutes from now    
